# पाठ १० - उत्पादनमा AI एजेन्टहरू

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरी AI एजेन्टहरूको **उत्पादन नमुना** सिक्नुहुनेछ। हामीले समावेश गरेका छौं:

- **अवलोकनशीलता** — एजेन्ट अन्तरक्रियाहरूमा समय निर्धारण र लगिङ थप्ने
- **मूल्यांकन** — प्रतिक्रिया गुणस्तरको स्कोर गर्न एक मूल्यांकन एजेन्ट प्रयोग गर्ने
- **लागत व्यवस्थापन** — टोकन अनुकूलन र मोडेल छनोटका रणनीतिहरू

परिदृश्य एउटा **यात्रा एजेन्ट** हो जसले प्रयोगकर्ताहरूलाई यात्रा योजनामा मद्दत गर्छ, जुन माथि नजरदारी र मूल्यांकन थपिएको छ।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचारहरू

एआई एजेन्टहरूलाई प्रोटोटाइपबाट उत्पादनमा सार्न तीन स्तम्भहरूमा सावधानीपूर्वक ध्यान दिनुपर्छ:

1. **अवलोकनीयता** — एजेन्ट के गर्दैछ, कति समय लाग्छ, र कुन उपकरणहरू कल गर्दछ भनेर तपाईंलाई देखिने हुनुपर्छ। ट्रेसिङ र लगिङ बिना, उत्पादन समस्याहरू डिबग गर्न लगभग असम्भव हुन्छ।

2. **मूल्याङ्कन** — स्वचालित गुणस्तर जाँचहरूले एजेन्टका प्रतिक्रिया समयसँगै सही, पूर्ण र सहयोगी रहन्छन् भन्ने सुनिश्चित गर्छ। मूल्याङ्कन गर्ने एजेन्टले परिभाषित मापदण्ड अनुसार प्रतिक्रियालाई स्कोर गर्न सक्छ।

3. **लागत व्यवस्थापन** — टोकन प्रयोगले सिधै लागतमा प्रभाव पार्छ। निम्ति रणनीतिहरू जस्तै प्रॉम्प्ट अप्टिमाइजेशन, मोडल चयन, र क्यासिङले गुणस्तर फाल्न नदिई खर्च नियन्त्रणमा मद्दत गर्छ।


## अवलोकनयोग्य एजेन्ट निर्माण गर्दै

हामी यात्रा उपकरणहरू परिभाषित गर्छौं र एजेन्ट कललाई टाइमिङको साथ र्याप गर्छौं ताकि हामी विलम्बता अनुगमन गर्न सकौं। उत्पादनमा तपाईं OpenTelemetry वा समान ट्रेसिंग ब्याकएन्डसँग एकीकरण गर्नुहुनेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन नमूनाहरू

एउटा सामान्य उत्पादन नमूना दोस्रो एजेन्टलाई **मूल्याङ्कनकर्ता** को रूपमा प्रयोग गर्नु हो। मूल्याङ्कनकर्ताले प्राथमिक एजेन्टको प्रतिक्रिया पूर्वनिर्धारित मापदण्डहरू जस्तै पूर्णता, शुद्धता, र सहयोगीता विरुद्ध स्कोर गर्दछ।

यसले सक्षम बनाउँछ:
- प्रतिक्रियाहरू प्रयोगकर्तासम्म पुग्नु अघि स्वचालित गुणस्तर ढोकाहरू
- जब प्रॉम्प्टहरू वा मोडेलहरू परिवर्तन हुन्छन् तब प्रतिगमन पत्ता लगाउने
- एजेन्ट प्रदर्शनको निरन्तर अनुगमन समयको साथ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत व्यवस्थापन रणनीतिहरू

उत्पादन एआई एजेन्टहरूको लागि लागत नियन्त्रण महत्त्वपूर्ण छ। यहाँ प्रमुख रणनीतिहरू छन्:

| रणनीति | विवरण |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | प्रणाली निर्देशनहरू संक्षिप्त राख्नुहोस्। इनपुट टोकन कमी गर्न अनावश्यक सन्दर्भ हटाउनुहोस्। |
    "| **मोडेल छनोट** | साधारण कार्यहरू जस्तै वर्गीकरण वा निष्कर्षणको लागि साना, सस्तो मोडेलहरू (जस्तै GPT-4.1-मिनी) प्रयोग गर्नुहोस्, र जटिल reasoning को लागि ठूलो मोडेल सुरक्षित राख्नुहोस्। |\n",
| **केशिंग** | उपकरण परिणामहरू र बारम्बार सोधिने प्रश्नहरू केश गर्नुहोस् ताकि दोहोरो API कलहरूबाट बच्न सकियोस्। |
| **टोकन बजेटहरू** | अपेक्षाकृत लामो उत्तरहरूलाई रोक्न `max_tokens` सीमा सेट गर्नुहोस्। |
| **ब्याचिङ** | जहाँ सम्भव हो, धेरै प्रयोगकर्ता प्रश्नलाई एउटै API कलमा समूहीकृत गर्नुहोस्। |

व्यावहारिक रूपमा, एक स्तरगत दृष्टिकोण राम्रो काम गर्छ: सरल अनुरोधहरूलाई छिटो, सानो मोडेलमा पठाउनुहोस् र जटिल प्रश्नहरूलाई मात्र सक्षम मोडेलमा बढाउनुहोस्।


## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. **एजेन्ट अन्तरक्रियामा अवलोकन क्षमता थप्ने**, समयक्रम र लगिङ्ग सहित, ट्रेसिङ र अनुगमनको आधार तयार गर्दै।
2. **एजेन्ट प्रतिक्रियाहरू स्वचालित रूपमा मूल्याङ्कन गर्ने** एक मूल्याङ्कन गर्ने एजेन्ट प्रयोग गरी जुन पूर्णता, शुद्धता, र उपयोगिताको स्कोर दिन्छ।
3. **खर्चहरूको व्यवस्थापन गर्ने** प्रम्प्ट अनुकूलन, मोडल छनौट, क्याचिङ्ग, र टोकन बजेट मार्फत।

यी उत्पादन ढाँचाहरूले तपाईंका AI एजेन्टहरूलाई भरपर्दो, मापनयोग्य, र लागत-कुशल बनाउन मद्दत गर्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
